In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import ccxt
import pandas as pd
import numpy as np
import ta
import time

from colorama import Fore, Style, init
from tabulate import tabulate

init(autoreset=True)


In [2]:
# ============================================================
# BINANCE CONNECTION
# ============================================================

exchange = ccxt.binance({
    'enableRateLimit': True,
    'options': {
        'defaultType': 'future'
    }
})


In [3]:
# ============================================================
# SETTINGS
# ============================================================

TIMEFRAME = '15m'

LIMIT = 300

TOP_VOLUME_COINS = 80

MIN_24H_VOLUME = 50_000_000

MIN_VOLATILITY = 1.0

DISPLACEMENT_MULTIPLIER = 1.2

MIN_FVG_COUNT = 3

MIN_DISPLACEMENT_COUNT = 3

SLEEP_TIME = 0.1

In [4]:
# ============================================================
# FETCH FUTURES USDT PAIRS
# ============================================================

def fetch_usdt_pairs():

    print(
        Fore.CYAN
        + "\nLoading Binance Futures Markets...\n"
    )

    markets = exchange.load_markets()

    symbols = []

    for symbol in markets:

        market = markets[symbol]

        try:

            if (
                market['quote'] == 'USDT'
                and market['active']
                and market['contract']
            ):

                symbols.append(symbol)

        except:
            pass

    print(
        Fore.GREEN
        + f"Found {len(symbols)} USDT Futures Pairs\n"
    )

    return symbols

In [5]:
# ============================================================
# FILTER HIGH VOLUME COINS
# ============================================================

def fetch_top_volume_symbols(symbols):

    print(
        Fore.CYAN
        + "Filtering High Volume Coins...\n"
    )

    tickers = exchange.fetch_tickers()

    filtered = []

    for symbol in symbols:

        try:

            ticker = tickers[symbol]

            quote_volume = ticker['quoteVolume']

            if quote_volume is None:
                continue

            if quote_volume >= MIN_24H_VOLUME:

                filtered.append({
                    'symbol': symbol,
                    'volume': quote_volume
                })

        except:
            pass

    filtered = sorted(
        filtered,
        key=lambda x: x['volume'],
        reverse=True
    )

    filtered = filtered[:TOP_VOLUME_COINS]

    final_symbols = [x['symbol'] for x in filtered]

    print(
        Fore.GREEN
        + f"Selected {len(final_symbols)} High Volume Coins\n"
    )

    return final_symbols


In [6]:
# ============================================================
# FETCH OHLCV DATA
# ============================================================

def fetch_ohlcv(symbol):

    try:

        ohlcv = exchange.fetch_ohlcv(
            symbol=symbol,
            timeframe=TIMEFRAME,
            limit=LIMIT
        )

        df = pd.DataFrame(
            ohlcv,
            columns=[
                'timestamp',
                'open',
                'high',
                'low',
                'close',
                'volume'
            ]
        )

        df['timestamp'] = pd.to_datetime(
            df['timestamp'],
            unit='ms'
        )

        return df

    except Exception as e:

        print(
            Fore.RED
            + f"Error Fetching {symbol}: {e}"
        )

        return None

In [7]:
# ============================================================
# ADD MOVING AVERAGES
# ============================================================

def add_moving_averages(df):

    df['ma25'] = ta.trend.ema_indicator(
        close=df['close'],
        window=25
    )

    return df


In [8]:
# ============================================================
# ADD ATR
# ============================================================

def add_atr(df):

    atr_indicator = ta.volatility.AverageTrueRange(
        high=df['high'],
        low=df['low'],
        close=df['close'],
        window=14
    )

    df['atr'] = atr_indicator.average_true_range()

    return df


In [9]:
# ============================================================
# ADD VOLATILITY
# ============================================================

def add_volatility(df):

    highest_price = (
        df['high']
        .rolling(20)
        .max()
    )

    lowest_price = (
        df['low']
        .rolling(20)
        .min()
    )

    df['volatility_percent'] = (
        (
            highest_price - lowest_price
        ) / df['close']
    ) * 100

    return df

In [10]:
# ============================================================
# DETECT FVG EXPANSION
# ============================================================

def detect_fvg_expansion(df):

    bullish_fvg_count = 0

    bearish_fvg_count = 0

    displacement_count = 0

    # ========================================================
    # CHECK LAST 5 CANDLES
    # ========================================================

    for i in range(-5, 0):

        current_high = df.iloc[i]['high']

        current_low = df.iloc[i]['low']

        previous2_high = df.iloc[i - 2]['high']

        previous2_low = df.iloc[i - 2]['low']

        # ====================================================
        # BULLISH FVG
        # ====================================================

        bullish_fvg = previous2_high < current_low

        # ====================================================
        # BEARISH FVG
        # ====================================================

        bearish_fvg = previous2_low > current_high

        # ====================================================
        # CANDLE RANGE
        # ====================================================

        candle_range = (
            current_high - current_low
        )

        avg_range = (
            (df['high'] - df['low'])
            .rolling(20)
            .mean()
            .iloc[i]
        )

        # ====================================================
        # DISPLACEMENT CANDLE
        # ====================================================

        displacement = (
            candle_range
            > avg_range * DISPLACEMENT_MULTIPLIER
        )

        # ====================================================
        # COUNT FVG
        # ====================================================

        if bullish_fvg:
            bullish_fvg_count += 1

        if bearish_fvg:
            bearish_fvg_count += 1

        # ====================================================
        # COUNT DISPLACEMENT
        # ====================================================

        if displacement:
            displacement_count += 1

    # ========================================================
    # ATR EXPANSION
    # ========================================================

    latest_atr = df.iloc[-1]['atr']

    previous_atr = df.iloc[-2]['atr']

    atr_rising = latest_atr > previous_atr

    # ========================================================
    # VOLATILITY EXPANSION
    # ========================================================

    latest_volatility = (
        df.iloc[-1]['volatility_percent']
    )

    high_volatility = (
        latest_volatility > MIN_VOLATILITY
    )

    # ========================================================
    # TREND STRUCTURE
    # ========================================================

    bullish_structure = (
        df.iloc[-1]['close']
        > df.iloc[-1]['ma25']
    )

    bearish_structure = (
        df.iloc[-1]['close']
        < df.iloc[-1]['ma25']
    )

    # ========================================================
    # FINAL BULLISH SIGNAL
    # ========================================================

    bullish_signal = (

        bullish_fvg_count >= MIN_FVG_COUNT

        and displacement_count >= MIN_DISPLACEMENT_COUNT

        and atr_rising

        and high_volatility

        and bullish_structure
    )

    # ========================================================
    # FINAL BEARISH SIGNAL
    # ========================================================

    bearish_signal = (

        bearish_fvg_count >= MIN_FVG_COUNT

        and displacement_count >= MIN_DISPLACEMENT_COUNT

        and atr_rising

        and high_volatility

        and bearish_structure
    )

    # ========================================================
    # RETURN SIGNALS
    # ========================================================

    if bullish_signal:

        return (
            "BULLISH_EXPANSION",
            bullish_fvg_count,
            displacement_count
        )

    elif bearish_signal:

        return (
            "BEARISH_EXPANSION",
            bearish_fvg_count,
            displacement_count
        )

    return None, 0, 0

In [11]:
# ============================================================
# SCAN MARKET
# ============================================================

def scan_market():

    print(
        Fore.YELLOW
        + "\nSTARTING INSTITUTIONAL EXPANSION SCANNER\n"
    )

    symbols = fetch_usdt_pairs()

    symbols = fetch_top_volume_symbols(symbols)

    results = []

    for index, symbol in enumerate(symbols):

        try:

            print(
                Fore.CYAN
                + f"[{index+1}/{len(symbols)}] Scanning {symbol}"
            )

            df = fetch_ohlcv(symbol)

            if df is None:
                continue

            # =================================================
            # ADD INDICATORS
            # =================================================

            df = add_moving_averages(df)

            df = add_atr(df)

            df = add_volatility(df)

            # =================================================
            # DETECT SIGNAL
            # =================================================

            signal, fvg_count, displacement_count = (
                detect_fvg_expansion(df)
            )

            # =================================================
            # SAVE RESULTS
            # =================================================

            if signal is not None:

                latest = df.iloc[-1]

                results.append({

                    'Symbol': symbol,

                    'Signal': signal,

                    'Price': round(
                        latest['close'],
                        4
                    ),

                    'ATR': round(
                        latest['atr'],
                        4
                    ),

                    'Volatility %': round(
                        latest['volatility_percent'],
                        2
                    ),

                    'FVG Count': fvg_count,

                    'Displacement Count': displacement_count
                })

            time.sleep(SLEEP_TIME)

        except Exception as e:

            print(
                Fore.RED
                + f"Scanner Error {symbol}: {e}"
            )

    return results

In [12]:
# ============================================================
# DISPLAY RESULTS
# ============================================================

def display_results(results):

    if len(results) == 0:

        print(
            Fore.RED
            + "\nNO EXPANSION SETUPS FOUND\n"
        )

        return

    df = pd.DataFrame(results)

    df = df.sort_values(
        by='FVG Count',
        ascending=False
    )

    print(
        Fore.GREEN
        + "\nEXPANSION SETUPS FOUND\n"
    )

    print(
        tabulate(
            df,
            headers='keys',
            tablefmt='fancy_grid',
            showindex=False
        )
    )


In [13]:
# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":

    results = scan_market()

    display_results(results)

    print(
        Fore.CYAN
        + "\nSCAN COMPLETED\n"
    )


STARTING INSTITUTIONAL EXPANSION SCANNER


Loading Binance Futures Markets...

Found 583 USDT Futures Pairs

Filtering High Volume Coins...

Selected 66 High Volume Coins

[1/66] Scanning BTC/USDT:USDT
[2/66] Scanning ETH/USDT:USDT
[3/66] Scanning HYPE/USDT:USDT
[4/66] Scanning SOL/USDT:USDT
[5/66] Scanning ZEC/USDT:USDT
[6/66] Scanning NEAR/USDT:USDT
[7/66] Scanning XAU/USDT:USDT
[8/66] Scanning XAG/USDT:USDT
[9/66] Scanning CL/USDT:USDT
[10/66] Scanning XRP/USDT:USDT
[11/66] Scanning EDEN/USDT:USDT
[12/66] Scanning BEAT/USDT:USDT
[13/66] Scanning BSB/USDT:USDT
[14/66] Scanning DOGE/USDT:USDT
[15/66] Scanning ONDO/USDT:USDT
[16/66] Scanning SUI/USDT:USDT
[17/66] Scanning BNB/USDT:USDT
[18/66] Scanning GENIUS/USDT:USDT
[19/66] Scanning FIDA/USDT:USDT
[20/66] Scanning WLD/USDT:USDT
[21/66] Scanning SNDK/USDT:USDT
[22/66] Scanning BZ/USDT:USDT
[23/66] Scanning PROVE/USDT:USDT
[24/66] Scanning 1000PEPE/USDT:USDT
[25/66] Scanning BILL/USDT:USDT
[26/66] Scanning ALT/USDT:USDT
[27/66] Scann